# Pluggable output writers

ECGProcess writes processed data through a registry of *writers*. A writer
is selected with the `writer` argument of `ECGTable.write_ecg`, either by
its registry key (e.g. `'npy'`, `'figure'`) or by passing a custom writer
object. The available keys live in `ecgprocess.writer.WRITERS`.

In this recipe we walk through each built-in writer:

* the tabular`table`,
* the bundled `npz`,
* the per-array `npy`,
* the `tensorflow` tfrecord,
* the `wfdb` PhysioNet record,
* the figure-based `figure` and `pixel` writers

Finally, we show how to plug in your own writer, first as a bare function and then as a registered `BaseWriter` subclass.

To get started we import the relevant functions and classes and load some example data.
Given the direct relationship with `ECGTable`, we will show each writer by directly applying using the table class. 

In [1]:
import io
import os
import importlib.util
import tempfile
import matplotlib
# a non-interactive backend for headless figure rendering
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image
from ecgprocess.tabular import ECGTable
from ecgprocess.writer import (
    WRITERS,
    BaseWriter,
    FigureWriter,
    PixelWriter,
    WFDBWriter,
    register_writer,
)
from ecgprocess.plot_ecgs import ECGStyle
from ecgprocess.process_dicom import ECGDICOMReader
from ecgprocess.example_data.examples import (
    parsed_config,
    list_dicom_paths,
)
# #### example DICOM files and their parsed config
dicom_path_list = [
    list_dicom_paths()['example_dicom_1'],
    list_dicom_paths()['example_dicom_2'],
]
parser_dicom = parsed_config()['parsed_dicom1']
# the registry keys accepted as `writer=` values
print(f'Available pre-defined writers: {sorted(WRITERS)}.')

Available pre-defined writers: ['figure', 'npy', 'npz', 'pixel', 'table', 'tensorflow', 'wfdb'].


## Flat tables

The `table` writer writes each extracted source to a (optionally) gzip-compressed, tab-delimited files:

* `metadata.tsv.gz`,
* `waveforms.tsv.gz`,
* `medianbeats.tsv.gz`.

The latter two in long format, so records of differing length need no padding. It is covered in depth in the [Scaled processing of ECG data](tabular.ipynb) recipe and shown here for
completeness.

In [2]:
with tempfile.TemporaryDirectory() as temp_dir:
    os.chdir(temp_dir)
    # #### the actual reading and writing part
    reader = ECGDICOMReader()
    ecgtable = ECGTable(reader, path_list=dicom_path_list)
    res = ecgtable(verbose=False).write_ecg(
        parsed_config=parser_dicom, write_failed=False, writer='table')
    # #### Now show files were created
    print('Directory content:')
    for file_name in sorted(os.listdir(getattr(res, 'target_path'))):
        print('   - {}'.format(file_name))

Directory content:
   - medianbeats.tsv.gz
   - metadata.tsv.gz
   - waveforms.tsv.gz


## NumPy archive 

The `npz` writer bundles every extracted source into a single NumPy `.npz` archive (`ecg_data.npz`), with the non-numeric metadata columns
dropped and remaining column names written to a `header_metadata.txt` sidecar.
Because the sources are stacked into arrays, we fix a common signal length first using `signal_length_w` and `signal_length_m`.

In [3]:
with tempfile.TemporaryDirectory() as temp_dir:
    os.chdir(temp_dir)
    # ### writing
    reader = ECGDICOMReader()
    # a common length lets the per-record signals stack into one array
    ecgtable = ECGTable(reader, path_list=dicom_path_list,
                        signal_length_w=5000, signal_length_m=600)
    res = ecgtable(verbose=False).write_ecg(
        parsed_config=parser_dicom, write_failed=False, writer='npz')
    # #### read these stored file in and confirm its content
    target = getattr(res, 'target_path')
    print('Directory content:')
    for file_name in sorted(os.listdir(target)):
        print('   - {}'.format(file_name))
    with np.load(os.path.join(target, 'ecg_data.npz')) as data:
        median = data['MedianBeats']
# (n_records, n_leads, n_samples)
median.shape

The following non-numerical columns were dropped when converting data to numpy arrays: {'key', 'Manufacturer', 'resolution unit (medianbeats)', 'sampling frequency unit', 'Softwave version', 'Model name', 'filename', 'resolution unit (waveforms)', 'unique identifier'}.


Directory content:
   - ecg_data.npz
   - header_metadata.txt


(2, 12, 600)

## Per-array NumPy files 

Because npy files can only include a single array, the `npy` writer stores each data source in its own file, creating:

* `metadata.npy`,
* `waveforms.npy`,
* `medianbeats.npy`.

The casting the signals to a configurable dtype (`float32` by default). 
Similar to `npz`, the `npy` writer can bundle multiple records in a single file (increasing the dimension of the single array).
Because NumPy holds only numerical data, non-numeric metadata columns are dropped and the remaining column names are written to a
`header_metadata.txt` sidecar.

In [4]:
with tempfile.TemporaryDirectory() as temp_dir:
    os.chdir(temp_dir)
    # ### reading and wring
    reader = ECGDICOMReader()
    # the lead-specific signals differ in length, so we fix a common
    # length (pad/truncate) before writing to arrays
    ecgtable = ECGTable(reader, path_list=dicom_path_list,
                        signal_length_w=5000, signal_length_m=600,)
    res = ecgtable(verbose=False).write_ecg(
        parsed_config=parser_dicom, write_failed=False, writer='npy')
    # ### Get the file from dist and show content. 
    target = getattr(res, 'target_path')
    print('Directory content:')
    for file_name in sorted(os.listdir(target)):
        print('   - {}'.format(file_name))
    # read the median beats back in before the temp dir disappears
    median = np.load(os.path.join(target, 'medianbeats.npy'),
                     allow_pickle=False)
# (n_records, n_leads, n_samples)
median.shape

The following non-numerical columns were dropped when converting data to numpy arrays: {'key', 'Manufacturer', 'resolution unit (medianbeats)', 'sampling frequency unit', 'Softwave version', 'Model name', 'filename', 'resolution unit (waveforms)', 'unique identifier'}.


Directory content:
   - header_metadata.txt
   - medianbeats.npy
   - metadata.npy
   - waveforms.npy


(2, 12, 600)

## TensorFlow records 

The `tensorflow` writer serialises the stacked arrays into binary `tfrecord` files (`ecg_data.tfrecord`), the native input format for
`tf.data` pipelines. 
Similar to `.npz` all three information sources are written to a single file, which can contain multiple records (e.g. individuals). 

TensorFlow is an optional dependency, so make sure to have `tensorflow` available in the environment (e.g. `pip install ecgprocess[tensorflow]`).

In [5]:
if importlib.util.find_spec('tensorflow') is not None:
    with tempfile.TemporaryDirectory() as temp_dir:
        os.chdir(temp_dir)
        # #### reading and wrting
        reader = ECGDICOMReader()
        ecgtable = ECGTable(reader, path_list=dicom_path_list,
                            signal_length_w=5000, signal_length_m=600)
        res = ecgtable(verbose=False).write_ecg(
            parsed_config=parser_dicom, write_failed=False,
            writer='tensorflow')
        # ##### read-in and show content
        print('Directory content:')
        for file_name in sorted(os.listdir(getattr(res, 'target_path'))):
            print('   - {}'.format(file_name))
else:
    print('tensorflow writer skipped: optional dependency not installed')

I0000 00:00:1784627219.623052   23707 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
The following non-numerical columns were dropped when converting data to numpy arrays: {'key', 'Manufacturer', 'resolution unit (medianbeats)', 'sampling frequency unit', 'Softwave version', 'Model name', 'filename', 'resolution unit (waveforms)', 'unique identifier'}.


Directory content:
   - ecg_data.tfrecord
   - header_metadata.txt


## WFDB records

The `wfdb` writer writes one PhysioNet WFDB record per ECG UID: a multiplexed, format-16 (int16) `.dat` signal file paired with a `.hea`
header file including user specified metadata (e.g. lead names, units, sampling frequency).

Metadata is not stacked into arrays like the other writers; instead each record's MetaData fields are written as  comment lines. 
By default only the original UID is written as a comment, pass a list of field names to the `meta_fields` parameters, or `'all'` to copy every metadata key for that record.
Unlike `npz`/`npy`/`tensorflow`, a WFDB record is single-source, single-record file, so it can only store one type of signal data in the `.dat` file. 

The `wfdb` package is an optional dependency, so make sure to have `wfdb` available in
the environment (e.g. `pip install ecgprocess[wfdb]`).

After writing, we read one record back with `wfdb.rdrecord` and print
its header fields to confirm the round-trip.

In [6]:
if importlib.util.find_spec('wfdb') is not None:
    import wfdb # needed to read the file back in, the writer imports this internally. 
    with tempfile.TemporaryDirectory() as temp_dir:
        os.chdir(temp_dir)
        reader = ECGDICOMReader()
        ecgtable = ECGTable(reader, path_list=dicom_path_list,
                            extract_median=False)
        # #### an explicit subset of metadata fields as .hea comments
        wfdb_writer_subset = WFDBWriter(meta_fields=[
            'Manufacturer', 'Model name', 'number of leads',
            'sampling frequency (original)', 'wave_channel_sens_0',
            'wave_channel_baseline_0', 'wave_channel_correctionfactor_0',
        ])
        res = ecgtable(verbose=False).write_ecg(
            parsed_config=parser_dicom, write_failed=False,
            writer=wfdb_writer_subset)
        # #### read one record's header back and show its comments
        target = getattr(res, 'target_path')
        hea_name = sorted(
            f for f in os.listdir(target) if f.endswith('.hea'))[0]
        record = wfdb.rdrecord(os.path.join(target, hea_name[:-4]))
    print('Header comments (subset meta_fields):')
    for comment in record.comments:
        print('   - {}'.format(comment))
else:
    print('wfdb writer skipped: optional dependency not installed')

Header comments (subset meta_fields):
   - UID: 1.3.6.1.4.1.40744.65.221835449869280278116555339248567804126
   - Manufacturer: GE
   - Model name: MAC55
   - number of leads: 12
   - sampling frequency (original): 500
   - wave_channel_sens_0: 5
   - wave_channel_baseline_0: 0
   - wave_channel_correctionfactor_0: 1


## Rendered figures 

The `figure` writer renders one ECG image per record through an `ECGDrawing` and saves it as `<UID>.<image_format>` (PNG by default).
It renders exactly one signal source, so in our example we will disable median-beat extraction with `extract_median=False`; extracting both waveforms and
median beats raises an error. 

In this example the signals are in microvolts, so we pass an `ECGStyle` scaled to a clinical print
(`y_lim=(-2000, 2000)`, `mm_per_mv=10/1000`) via `kwargs_call`; anything `ECGDrawing.__call__` accepts can be forwarded there.
See the [ECGDrawing](plot_ecgs.ipynb) examples in its own dedicated notebook.

In [7]:
# Set-up plotting callable 
style = ECGStyle(y_lim=(-2000, 2000), mm_per_mv=10 / 1000)
fig_writer = FigureWriter(kwargs_call={
    'style': style,
    'sampling_rate': 500,
    'show_lead_labels': True,
    'cal_pulse': 'square',
})
with tempfile.TemporaryDirectory() as temp_dir:
    os.chdir(temp_dir)
    # #### Reading and writing
    reader = ECGDICOMReader()
    # the figure writer renders exactly one source
    ecgtable = ECGTable(reader, path_list=dicom_path_list,
                        extract_median=False)
    res = ecgtable(verbose=False).write_ecg(
        parsed_config=parser_dicom, write_failed=False, writer=fig_writer)
    # #### show figure and directory content
    target = getattr(res, 'target_path')
    print('Directory content:')
    for file_name in sorted(os.listdir(target)):
        print('   - {}'.format(file_name))
    # read one rendered image into memory before the temp dir disappears
    png_name = sorted(f for f in os.listdir(target) if f.endswith('.png'))[1]
    with open(os.path.join(target, png_name), 'rb') as fh:
        png_bytes = fh.read()
Image(data=png_bytes)

Directory content:
   - 1.3.6.1.4.1.40744.65.221835449869280278116555339248567804126.png
   - 2.25.269796857626990821315969488216511468638.png


## Stacked pixel array

The pixel writer renders each record exactly like the figure writer, but instead of an image file it rasterises the figures and stacks them into a single
`(n-records, height, width, channels)` NHWC tensor saved as `figures.npy`, with a row-aligned `metadata.tsv`. 

For this example we render a single source (the waveforms), where `channels='gray'` keeps a single channel axis, and the figure `style` defined above is reused for scaling.
We also pass an `annotate` spec so the ECG calibration (`25 mm/s`,`10 mm/mV`) is drawn onto each figure *before* it is rasterised, becoming
part of the tensor - for the current example we increase the fontsize to make sure the annotation remains readable. 

Finally we read `figures.npy` back from disk and plot the first record to confirm the round-trip.

In [8]:
# annotate each figure with the ECG calibration before rasterising, so
# the unit label becomes part of the stored pixel tensor
pixel_writer = PixelWriter(
    channels='gray',
    kwargs_call={'style': style, 'sampling_rate': 500},
    annotate=[{'text': '25 mm/s   10 mm/mV', 'xy': (0.15, 0.08), 'fontsize' : 20}],
)
with tempfile.TemporaryDirectory() as temp_dir:
    os.chdir(temp_dir)
    # ### Reading and writing
    reader = ECGDICOMReader()
    ecgtable = ECGTable(reader, path_list=dicom_path_list,
                        extract_median=False)
    res = ecgtable(verbose=False).write_ecg(
        parsed_config=parser_dicom, write_failed=False, writer=pixel_writer)
    # #### reading the save tensor
    target = getattr(res, 'target_path')
    print('Directory content:')
    for file_name in sorted(os.listdir(target)):
        print('   - {}'.format(file_name))
    # read the stacked tensor back from disk
    tensor = np.load(os.path.join(target, 'figures.npy'), allow_pickle=False)
print('tensor shape (n_records, height, width, channels):', tensor.shape)
print('')

# #### plot the record back from the tensor, at native resolution
# (no intervening Figure/dpi/figsize, so this is a pixel-for-pixel view
# matching the figure writer's raw PNG in the cell above)
buf = io.BytesIO()
plt.imsave(buf, tensor[0, :, :, 0], cmap='gray', format='png')
Image(data=buf.getvalue())

Directory content:
   - figures.npy
   - metadata.tsv
tensor shape (n_records, height, width, channels): (2, 6000, 12000, 1)



## Bringing your own writer

The `writer` argument also accepts user-supplied writers, so new output formats need no changes to library code. 

The simplest form is a bare
callable matching the `BaseWriter.write` signature
`(meta_list, wave_list, median_list, ctx)`, where the three lists hold
the extracted metadata, waveform, and median-beat records for the chunk.

The final argument, `ctx`, is a `WriteContext` (from
`ecgprocess.writer`) that carries everything a writer needs from
`ECGTable`:

- `ctx.target` - the output directory to write into;
- `ctx.key_list` - the per-record unique identifiers (UIDs), aligned
  element-by-element with the three record lists;
- `ctx.e_meta`, `ctx.e_wave`, `ctx.e_median` - which sources were
  extracted;
- `ctx.chunk_num` - the current chunk number (`None` when all records
  are written in one go), appended to the standard filenames;
- `ctx.skip_data` and `ctx.failed_records` - bookkeeping for records a
  writer chooses to skip rather than fail on.

Here a small function writes a one-line-per-record summary of how many
leads each waveform carries.

In [9]:
def summary_writer(meta_list, wave_list, median_list, ctx):
    """Write a per-record lead-count summary to `summary.txt`."""
    lines = []
    for uid, wave in zip(ctx.key_list, wave_list, strict=True):
        n_leads = sum(arr is not None for arr in wave.values())
        lines.append('{}\t{} leads'.format(uid, n_leads))
    with open(os.path.join(ctx.target, 'summary.txt'), 'w') as fh:
        fh.write('\n'.join(lines) + '\n')

with tempfile.TemporaryDirectory() as temp_dir:
    os.chdir(temp_dir)
    reader = ECGDICOMReader()
    ecgtable = ECGTable(reader, path_list=dicom_path_list)
    # pass the function directly as the writer
    res = ecgtable(verbose=False).write_ecg(
        parsed_config=parser_dicom, write_failed=False,
        writer=summary_writer)
    with open(os.path.join(getattr(res, 'target_path'), 'summary.txt')) as fh:
        print(fh.read())

2.25.269796857626990821315969488216511468638	12 leads
1.3.6.1.4.1.40744.65.221835449869280278116555339248567804126	12 leads



For a reusable, named format, subclass `BaseWriter` and register it
with the `register_writer` decorator. Registration adds the class to
`WRITERS` under a key, so it can then be selected with `writer='<key>'`
exactly like a built-in writer.

Rather than hard-code an output name, the writer below reuses
`BaseWriter.file_names`, which returns the standard per-chunk filenames
(with the writer's `FILE_EXT` extension and any chunk suffix). We take
the first of these, the metadata filename (`metadata.tsv`), because the
lead count is one tabular row of metadata per record. Writing it under
the standard metadata name means the output follows the same naming and
chunking scheme as the built-in writers.

In [10]:
@register_writer('lead_count')
class LeadCountWriter(BaseWriter):
    """Write the per-record non-empty lead count as a `.tsv` table."""
    # the extension appended to the standard filenames
    FILE_EXT = '.tsv'
    def write(self, meta_list, wave_list, median_list, ctx):
        """Write one `metadata.tsv` row per record."""
        # the standard metadata filename for this chunk, e.g.
        # `metadata.tsv` (or `metadata_0.tsv` when chunked)
        meta_file, *_ = self.file_names(chunk_num=ctx.chunk_num)
        rows = ['uid\tn_leads']
        for uid, wave in zip(ctx.key_list, wave_list, strict=True):
            n_leads = sum(arr is not None for arr in wave.values())
            rows.append('{}\t{}'.format(uid, n_leads))
        with open(os.path.join(ctx.target, meta_file), 'w') as fh:
            fh.write('\n'.join(rows) + '\n')

# Now using lead_count writer and reading this back in using with.
with tempfile.TemporaryDirectory() as temp_dir:
    os.chdir(temp_dir)
    reader = ECGDICOMReader()
    ecgtable = ECGTable(reader, path_list=dicom_path_list)
    res = ecgtable(verbose=False).write_ecg(
        parsed_config=parser_dicom, write_failed=False,
        writer='lead_count')
    with open(os.path.join(getattr(res, 'target_path'),
                           'metadata.tsv')) as fh:
        print(fh.read())

uid	n_leads
2.25.269796857626990821315969488216511468638	12
1.3.6.1.4.1.40744.65.221835449869280278116555339248567804126	12

